# DSS Analisis Kemiripan Logo Merek Dagang
**Ghania Fazila (41122100060) | Universitas Widyatama**

## Urutan menjalankan (SETIAP SESSION BARU)
1. Cell 1 — Mount Drive
2. Cell 2 — Install library
3. Cell 3 — Setup & patch (jalankan SEKALI)
4. Cell 4 — Jalankan Streamlit (dapat URL)

**Update app.py:** edit di Drive → Cell 5
**Ada error:** Cell 6

## Cell 1 — Mount Google Drive

In [8]:
# ═══════════════════════════════════════════════
# CELL 1 — Mount Drive & Verifikasi
# ═══════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_BASE = '/content/drive/MyDrive/project_skripsi'

required = [
    f'{DRIVE_BASE}/Models/hybrid_best.pt',
    f'{DRIVE_BASE}/Models/pdki_embeddings.npy',
    f'{DRIVE_BASE}/Models/pdki_hsv.npy',
    f'{DRIVE_BASE}/Models/pdki_metadata.json',
    f'{DRIVE_BASE}/App/app.py',
]

print('Verifikasi file di Drive:')
for p in required:
    exists = os.path.exists(p)
    size   = os.path.getsize(p)/1024/1024 if exists else 0
    print(f'  {"OK" if exists else "MISSING"} {os.path.basename(p)} ({size:.1f} MB)')

pdki = Path(f'{DRIVE_BASE}/Datasets/PDKI')
n_img = sum(1 for _ in pdki.rglob('*.jpg')) if pdki.exists() else 0
print(f'  {"OK" if pdki.exists() else "MISSING"} Datasets/PDKI/ ({n_img:,} gambar)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Verifikasi file di Drive:
  OK hybrid_best.pt (107.0 MB)
  OK pdki_embeddings.npy (6.5 MB)
  OK pdki_hsv.npy (0.8 MB)
  OK pdki_metadata.json (3.5 MB)
  OK app.py (0.0 MB)
  OK Datasets/PDKI/ (6,698 gambar)


## Cell 2 — Install Dependencies

In [9]:
# ═══════════════════════════════════════════════
# CELL 2 — Install Dependencies
# ═══════════════════════════════════════════════
!pip install -q streamlit pyngrok scikit-image grad-cam 2>&1 | tail -3

import streamlit, torch
print(f'streamlit : {streamlit.__version__}')
print(f'torch     : {torch.__version__}')
print(f'CUDA      : {torch.cuda.is_available()}')

streamlit : 1.59.1
torch     : 2.11.0+cpu
CUDA      : False


## Cell 3 — Setup & Patch (jalankan SEKALI per session)

In [10]:
# ═══════════════════════════════════════════════
# CELL 3 — Setup & Patch (jalankan SEKALI per session)
# ═══════════════════════════════════════════════
import os, shutil, json
import pandas as pd
from pathlib import Path
from collections import Counter

DRIVE_BASE = '/content/drive/MyDrive/project_skripsi'
APP_DIR    = '/content/DSS_App'
PDKI_DIR   = f'{DRIVE_BASE}/Datasets/PDKI'
os.makedirs(APP_DIR, exist_ok=True)

# 1. Copy semua file
print('1. Copy file...')
for src, dst in [
    (f'{DRIVE_BASE}/Models/hybrid_best.pt',      f'{APP_DIR}/hybrid_best.pt'),
    (f'{DRIVE_BASE}/Models/pdki_embeddings.npy', f'{APP_DIR}/pdki_embeddings.npy'),
    (f'{DRIVE_BASE}/Models/pdki_hsv.npy',        f'{APP_DIR}/pdki_hsv.npy'),
    (f'{DRIVE_BASE}/Models/pdki_metadata.json',  f'{APP_DIR}/pdki_metadata.json'),
    (f'{DRIVE_BASE}/App/app.py',                 f'{APP_DIR}/app.py'),
]:
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'   OK {os.path.basename(dst)} ({os.path.getsize(dst)/1024/1024:.1f} MB)')
    else:
        print(f'   MISSING {src}')

# 2. Update metadata dari CSV
print('\n2. Update metadata nama_merek dari CSV...')
meta_lookup = {}
for csv_path in Path(PDKI_DIR).rglob('*.csv'):
    if 'batch_summary' in csv_path.name: continue
    try:
        df = pd.read_csv(csv_path)
        if 'application_id' not in df.columns: continue
        fkey = csv_path.stem
        for _, row in df.iterrows():
            fname = Path(str(row.get('image_filename',''))).name
            if fname: meta_lookup[(fkey, fname)] = row.to_dict()
    except: pass

with open(f'{APP_DIR}/pdki_metadata.json', encoding='utf-8') as f:
    all_meta = json.load(f)

n_ok = 0
for e in all_meta:
    orig  = e.get('path','')
    fname = e.get('filename','')
    fkey  = orig.split('PDKI/')[-1].split('/')[0] if 'PDKI/' in orig else None
    key   = (fkey, fname) if fkey else None
    if key and key in meta_lookup:
        r = meta_lookup[key]
        e.update({
            'nama_merek'        : str(r.get('nama_merek',''))[:60],
            'brand'             : str(r.get('nama_merek', fkey))[:60],
            'application_id'    : str(r.get('application_id','')),
            'nomor_permohonan'  : str(r.get('nomor_permohonan','')),
            'kelas_nice'        : str(r.get('kelas_nice','')),
            'tahun'             : str(r.get('tahun','')),
            'tanggal_permohonan': str(r.get('tanggal_permohonan','')),
            'status_permohonan' : str(r.get('status_permohonan','N/A')),
            'owner_name'        : str(r.get('owner_name',''))[:60],
            'class_desc'        : str(r.get('class_desc',''))[:100],
        })
        n_ok += 1

with open(f'{APP_DIR}/pdki_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(all_meta, f, indent=2, ensure_ascii=False)
print(f'   OK {n_ok}/{len(all_meta)} entri diupdate')

# 3. Patch app.py
print('\n3. Patch app.py...')
with open(f'{APP_DIR}/app.py', 'r', encoding='utf-8') as f:
    c = f.read()

# Patch IMAGES_DIR
c = c.replace(
    'IMAGES_DIR    = BASE_DIR / "pdki_images"',
    f'IMAGES_DIR    = Path("{PDKI_DIR}")'
)

# Patch exclude self
OLD = ('    final_scores  = alpha * hybrid_scores + (1 - alpha) * color_scores\n'
       '    top_idx       = np.argsort(final_scores)[::-1][:k]')
NEW = ('    final_scores  = alpha * hybrid_scores + (1 - alpha) * color_scores\n'
       '    # Exclude query dari hasil (diri sendiri, score ~1.0)\n'
       '    if final_scores.max() > 0.9999:\n'
       '        final_scores[final_scores.argmax()] = -1\n'
       '    top_idx       = np.argsort(final_scores)[::-1][:k]')
if OLD in c:
    c = c.replace(OLD, NEW)
    print('   OK exclude self di-patch')
else:
    print('   INFO exclude self sudah ada / tidak perlu di-patch')

with open(f'{APP_DIR}/app.py', 'w', encoding='utf-8') as f:
    f.write(c)

print('\nSetup selesai. Lanjut ke Cell 4.')

1. Copy file...
   OK hybrid_best.pt (107.0 MB)
   OK pdki_embeddings.npy (6.5 MB)
   OK pdki_hsv.npy (0.8 MB)
   OK pdki_metadata.json (3.5 MB)
   OK app.py (0.0 MB)

2. Update metadata nama_merek dari CSV...
   OK 6698/6698 entri diupdate

3. Patch app.py...
   INFO exclude self sudah ada / tidak perlu di-patch

Setup selesai. Lanjut ke Cell 4.


## Cell 4 — Jalankan Streamlit
**Ganti `NGROK_TOKEN` dengan authtoken dari https://dashboard.ngrok.com/get-started/your-authtoken**

In [11]:
# ═══════════════════════════════════════════════
# CELL 4 — Jalankan Streamlit
# Ganti NGROK_TOKEN dengan token dari dashboard.ngrok.com
# ═══════════════════════════════════════════════
import subprocess, time
from pyngrok import ngrok, conf

APP_DIR     = '/content/DSS_App'
NGROK_TOKEN = '3G5dWA2BCHpYeHmkFiIUkmngZU0_tx9wM8gxpvesZ3SWVLfA'  # ← GANTI INI

!pkill -f streamlit 2>/dev/null || true
ngrok.kill()
time.sleep(2)

conf.get_default().auth_token = NGROK_TOKEN

subprocess.Popen(
    ['streamlit', 'run', f'{APP_DIR}/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false',
     '--browser.gatherUsageStats', 'false'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

print('Menunggu Streamlit start...')
time.sleep(8)

tunnel = ngrok.connect(8501)
print('\n' + '='*50)
print('DSS SIAP DIAKSES')
print('='*50)
print(f'URL: {tunnel.public_url}')
print('='*50)

^C
Menunggu Streamlit start...

DSS SIAP DIAKSES
URL: https://broadness-lurch-deletion.ngrok-free.dev


## Cell 5 (Opsional) — Update app.py & Restart
Jalankan setelah edit `app.py` di Drive.

In [7]:
# ═══════════════════════════════════════════════
# CELL 5 — Update app.py & Restart
# Jalankan setelah edit app.py di Drive
# ═══════════════════════════════════════════════
import shutil, subprocess, time
from pyngrok import ngrok, conf

DRIVE_BASE  = '/content/drive/MyDrive/project_skripsi'
APP_DIR     = '/content/DSS_App'
PDKI_DIR    = f'{DRIVE_BASE}/Datasets/PDKI'
NGROK_TOKEN = '3G5dWA2BCHpYeHmkFiIUkmngZU0_tx9wM8gxpvesZ3SWVLfA'  # ← sama dengan Cell 4

shutil.copy2(f'{DRIVE_BASE}/App/app.py', f'{APP_DIR}/app.py')

with open(f'{APP_DIR}/app.py', 'r', encoding='utf-8') as f:
    c = f.read()
c = c.replace('IMAGES_DIR    = BASE_DIR / "pdki_images"',
              f'IMAGES_DIR    = Path("{PDKI_DIR}")')
OLD = ('    final_scores  = alpha * hybrid_scores + (1 - alpha) * color_scores\n'
       '    top_idx       = np.argsort(final_scores)[::-1][:k]')
NEW = ('    final_scores  = alpha * hybrid_scores + (1 - alpha) * color_scores\n'
       '    if final_scores.max() > 0.9999:\n'
       '        final_scores[final_scores.argmax()] = -1\n'
       '    top_idx       = np.argsort(final_scores)[::-1][:k]')
if OLD in c: c = c.replace(OLD, NEW)
with open(f'{APP_DIR}/app.py', 'w', encoding='utf-8') as f:
    f.write(c)
print('OK app.py diupdate dan di-patch')

!pkill -f streamlit 2>/dev/null || true
ngrok.kill()
time.sleep(3)

conf.get_default().auth_token = NGROK_TOKEN
subprocess.Popen(
    ['streamlit', 'run', f'{APP_DIR}/app.py',
     '--server.port', '8501', '--server.headless', 'true',
     '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(8)
tunnel = ngrok.connect(8501)
print(f'App di-restart. URL: {tunnel.public_url}')

OK app.py diupdate dan di-patch
^C
App di-restart. URL: https://broadness-lurch-deletion.ngrok-free.dev


## Cell 6 (Opsional) — Debug
Jalankan kalau app tidak bisa dibuka / ada error

In [ ]:
# ═══════════════════════════════════════════════
# CELL 6 — Debug
# Jalankan kalau app tidak bisa dibuka / ada error
# ═══════════════════════════════════════════════
!streamlit run /content/DSS_App/app.py \
    --server.port 8502 \
    --server.headless true \
    2>&1 | head -60